# Phase 12 - Die FFN-Zwischenschicht: die ausgelassene Komponente

**Braucht eine A100**, ~20 min.

Inventur dieser Sitzung: Residuum (ausgiebig), Attention-Kanten (Cell 29/29b), MoE-
**Router** (Phase 9/9b/10, inkl. kausaler Ablation), Unembedding (beide Linsen) - und
die **FFN-Zwischenschicht: nie**. In Phase 9/9b/10 kommt `router` vor, aber kein
einziges `gate_proj`/`up_proj`/`down_proj`. Wir haben immer nur gefragt, *welcher*
Experte feuert, nie *was in ihm passiert*.

Das ist die interessanteste Auslassung: sie ist die breiteste Aktivierung pro Token,
sie liegt **hinter** der Nichtlinearitaet (SwiGLU druckt viele Einheiten gegen null,
sie ist also von Natur aus duennbesetzt) - und damit ist sie genau das, was ein SAE
nachbaut, nur vom Modell selbst gelernt statt von uns hinzutrainiert.

## Eine Einschraenkung legt das Design fest

Innerhalb eines festen Praefixes ist der Vorwaertszustand **deterministisch**. Zwei
Ziehungen mit identischen ersten Zeichen haben exakt dieselben Aktivierungen - dort
ist nichts zu korrelieren. Der Zustand legt eine *Wahrscheinlichkeit* fest, die
Ziehung wuerfelt. Die Frage muss lauten: **was im Zustand bestimmt die
Wahrscheinlichkeit?** Dafuer muss der Praefix variieren - und die Zeitanalyse hat
gezeigt, wo: bei Zeichen 90-130, an der ersten Datenzelle, wo Praefixe wie
`| Service (Local Name) | Storage Limits ` (7 von 9) und
`| Service Local Name | Storage Limits | ` (1 von 8) stark trennen. Beide englisch,
beide vor der Entscheidung.

## Ablauf

0. **Architektur auslesen** - alle echten Weiten, keine Schaetzung
1. Praefixe ernten (96 Ziehungen, nur selbst-nicht-gekippte)
2. je Praefix erzwingen und die Rate messen -> Zustaende **mit** Zielgroesse
3. je Praefix ein Vorwaertspass ueber den KV-Cache mit **genau einem Token**, damit
   jede aufgezeichnete Zeile eindeutig zu dieser Position gehoert
4. Einheiten waehlen, die hohe von niedrigen Raten trennen
5. **der eigentliche Test**: diese Einheiten auf null setzen und die Rate neu messen,
   gegen gleich viele zufaellige **aktive** Einheiten als Kontrolle

Schritt 4 ist bei ~8 Praefixen explorativ - mit acht Punkten sieht fast jede Einheit
signifikant aus, und ich habe in dieser Sitzung dreimal vorgefuehrt, wie leicht man
sich dabei etwas einredet. Schritt 5 ist der Grund, warum es trotzdem taugt: **eine
ablatierte Einheit, die die Rate bewegt, braucht keinen Permutationstest.** Die
Richtung ist vorregistriert - gewaehlt werden Einheiten, die in *hohen* Raten *hoch*
sind, ihre Ablation muss die Rate *senken*. Ein Wirkungs-Tor prueft vorher, dass die
Haken die Logits ueberhaupt veraendern.


In [ ]:
# === PHASE 12 - DIE FFN-ZWISCHENSCHICHT: DIE AUSGELASSENE KOMPONENTE =======
# Inventur der Sitzung: Residuum (ausgiebig), Attention-Kanten (Cell 29/29b),
# MoE-ROUTER (Phase 9/9b/10, inkl. Ablation), Unembedding (beide Linsen) -
# und die FFN-Zwischenschicht: NIE. In Phase 9/9b/10 kommt 'router' vor, aber
# kein einziges gate_proj/up_proj/down_proj. Wir haben immer nur gefragt,
# WELCHER Experte feuert, nie WAS IN IHM passiert.
#
# Das ist die interessanteste Auslassung, aus drei Gruenden:
#  * sie ist die breiteste Aktivierung pro Token (8 aktive Experten parallel)
#  * sie liegt HINTER der Nichtlinearitaet - bei SwiGLU druckt das Tor viele
#    Einheiten gegen null, sie ist also von Natur aus duennbesetzt
#  * damit ist sie genau das, was ein SAE nachbaut - nur vom Modell selbst
#    gelernt statt von uns hinzutrainiert
#
# EINE EINSCHRAENKUNG LEGT DAS DESIGN FEST: innerhalb eines festen Praefixes
# ist der Vorwaertszustand DETERMINISTISCH. Zwei Ziehungen mit identischen
# ersten Zeichen haben exakt dieselben Aktivierungen - dort ist nichts zu
# korrelieren. Der Zustand legt eine WAHRSCHEINLICHKEIT fest, die Ziehung
# wuerfelt. Die Frage muss also lauten: was im Zustand bestimmt die
# Wahrscheinlichkeit? Dafuer muss der PRAEFIX variieren.
#
# Die Zeitanalyse hat gezeigt, wo: die Entscheidung faellt bei Zeichen 90-130,
# an der ersten Datenzelle. Und die Praefixe dort trennen stark:
#   '| Service (Local Name) | Storage Limits '  7 von 9 kippen
#   '| Service Local Name | Storage Limits | '  1 von 8
# Beide englisch, beide vor der Entscheidung.
#
# ABLAUF
#  0 Architektur auslesen - alle echten Weiten, keine Schaetzung von mir
#  1 Praefixe ernten: Original-Prompt N mal, verschiedene Anfaenge sammeln
#  2 je Praefix erzwingen und die Rate messen -> Zustaende MIT Zielgroesse
#  3 je Praefix EINEN Vorwaertspass, Router und FFN-Zwischenschicht
#    aufzeichnen - ueber den KV-Cache mit genau EINEM Token, damit jede
#    aufgezeichnete Zeile eindeutig zu dieser Position gehoert
#  4 Einheiten waehlen, die hohe von niedrigen Raten trennen
#  5 DER EIGENTLICHE TEST: diese Einheiten auf null setzen und die Rate NEU
#    messen, gegen eine gleich grosse Zufallsauswahl als Kontrolle
#
# Schritt 4 ist bei ~8 Praefixen explorativ - mit acht Punkten sieht fast jede
# Einheit signifikant aus. Schritt 5 ist der Grund, warum es trotzdem taugt:
# eine ablatierte Einheit, die die Rate bewegt, braucht keinen Permutationstest.
# VORREGISTRIERT: die gewaehlten Einheiten sind die, die in HOHEN Raten HOCH
# sind. Ihre Ablation muss die Rate SENKEN. Die Zufallsauswahl darf es nicht.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, unicodedata, random
import numpy as np, glob, json, gc, sys, time
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig)."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_ffn")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
PTES=set("nome nomes servico servicos armazenamento limite limites preco mes gratuito "
         "conta cada para com uma nao mais seu sua nombre servicio servicios "
         "almacenamiento precio cuenta los las del con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _fremd(s):
    return [c for c in s if c.isalpha() and ord(c)>=0x250
            and any(a<=ord(c)<=b for a,b in FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def _entakz(s):
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[c for c in t if c.isalpha()]; fo=_fremd(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def classify_breit(t):
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for c2 in t if c2.isalpha() and 0xC0<=ord(c2)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW=("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
def sauber(p):
    """Ein Praefix taugt nur, wenn er selbst NICHT schon gekippt ist - sonst
       misst man die eigene Vorgabe."""
    return classify_breit(p)=="english" and len(p.strip())>0
def ernte_praefixe(texte,laenge=40,mindest=4,hoechstens=10):
    """Verschiedene Antwortanfaenge sammeln, nach Haeufigkeit. Nur saubere."""
    g=collections.Counter(t[:laenge] for t in texte if len(t)>=laenge)
    aus=[(p,n) for p,n in g.most_common() if n>=mindest and sauber(p)]
    return aus[:hoechstens]
def wilson(k,n,z=1.96):
    if n==0: return (0.,0.,0.)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def fisher2x2(a,b,c,d,einseitig=False):
    from math import lgamma,exp
    lf=lambda n: lgamma(n+1); n=a+b+c+d
    def pr(x):
        y=a+b-x; z=a+c-x; w=n-x-y-z
        if min(y,z,w)<0: return 0.0
        return exp(lf(a+b)+lf(c+d)+lf(a+c)+lf(b+d)-lf(n)-lf(x)-lf(y)-lf(z)-lf(w))
    hi=min(a+b,a+c)
    if einseitig: return min(1.0,sum(pr(x) for x in range(a,hi+1)))
    p0=pr(a)
    return min(1.0,sum(pr(x) for x in range(0,hi+1) if pr(x)<=p0*(1+1e-9)))
def logit(p,eps=1e-6):
    p=min(max(p,eps),1-eps); return math.log(p/(1-p))
def trenn_statistik(A,y):
    """A: (n_praefixe, n_einheiten). y: Rate je Praefix. Liefert je Einheit
       eine standardisierte Differenz zwischen hoher und niedriger Haelfte.
       Positiv = in HOHEN Raten hoeher."""
    y=np.asarray(y,float); med=float(np.median(y))
    hi=y>med; lo=~hi
    if hi.sum()<2 or lo.sum()<2: return None
    mh=A[hi].mean(0); ml=A[lo].mean(0)
    sd=np.sqrt((A[hi].var(0,ddof=1)+A[lo].var(0,ddof=1))/2.0)+1e-8
    return (mh-ml)/sd
def waehle_einheiten(d,k):
    """die k Einheiten mit der groessten POSITIVEN Trennung - Richtung ist
       vorregistriert: hoch in hohen Raten, Ablation muss senken"""
    return list(np.argsort(-d)[:k])
def urteil_ffn(arch_ok,n_praefix,k_basis,n_basis,k_abl,n_abl,k_zuf,n_zuf,alpha=0.05):
    if not arch_ok: return "ARCHITEKTUR-NICHT-GEFUNDEN"
    if n_praefix<4: return "ZU-WENIG-PRAEFIXE"
    senkt=lambda k,n: (k/n<k_basis/n_basis) and fisher2x2(k,n-k,k_basis,n_basis-k_basis)<alpha
    a=senkt(k_abl,n_abl); z=senkt(k_zuf,n_zuf)
    if a and not z: return "EINHEITEN-KAUSAL"
    if a and z:     return "UNSPEZIFISCH"
    return "KEIN-EFFEKT"
# ---------------- Ausfuehrung ------------------------------------------------
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
N_ERNTE=int(globals().get("N_ERNTE",96)); N_PRAEF=int(globals().get("N_PRAEF",64))
MAX_NEW=int(globals().get("MAX_NEW",64)); CHUNK=int(globals().get("CHUNK",16))
TEMP=float(globals().get("TEMP",1.0)); SEED=int(globals().get("SEED",20260807))
K_EINH=int(globals().get("K_EINH",64)); PLAENGE=int(globals().get("PLAENGE",24))
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
BASIS=prompt_text(PROMPTS[ZIEL_ID])
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
# ---------------- 0  Architektur --------------------------------------------
print("="*80); print("ARCHITEKTUR - ausgelesen, nicht geschaetzt"); print("="*80)
cfg=model.config
def zeig(*n):
    for x in n:
        v=getattr(cfg,x,None)
        if v is not None: print("  %-28s %s"%(x,v))
zeig("model_type","hidden_size","num_hidden_layers","num_attention_heads",
     "num_key_value_heads","head_dim","intermediate_size","moe_intermediate_size",
     "num_experts","num_experts_per_tok","shared_expert_intermediate_size",
     "n_shared_experts","vocab_size","linear_num_key_heads","linear_num_value_heads",
     "linear_key_head_dim","linear_value_head_dim","full_attention_interval")
RX_EXP=re.compile(r"^model\.layers\.(\d+)\.mlp\.experts\.(\d+)\.down_proj$")
RX_GATE=re.compile(r"^model\.layers\.(\d+)\.mlp\.gate$")
EXPERTEN={}; GATES={}
for nm,mod in model.named_modules():
    m=RX_EXP.match(nm)
    if m: EXPERTEN.setdefault(int(m.group(1)),{})[int(m.group(2))]=mod
    m=RX_GATE.match(nm)
    if m: GATES[int(m.group(1))]=mod
ARCH_OK=bool(EXPERTEN)
print("")
if ARCH_OK:
    L0=sorted(EXPERTEN)[0]
    ein=EXPERTEN[L0][0].in_features
    print("  MoE-Schichten gefunden      %d (Nummern %d..%d)"
          %(len(EXPERTEN),min(EXPERTEN),max(EXPERTEN)))
    print("  Experten je Schicht         %d"%len(EXPERTEN[L0]))
    print("  FFN-Zwischenbreite/Experte  %d  (Residuum: %d)"%(ein,cfg.hidden_size))
    akt=getattr(cfg,"num_experts_per_tok",8)
    print("  aktiv je Token              %d Experten x %d = %d Einheiten = %.1f x Residuum"
          %(akt,ein,akt*ein,akt*ein/cfg.hidden_size))
    print("  Router-Gates gefunden       %d"%len(GATES))
else:
    print("  KEINE Experten-Module unter dem erwarteten Namen gefunden.")
    print("  Vorhandene mlp-Module (Auszug), damit der naechste Anlauf trifft:")
    for nm,_ in list(model.named_modules()):
        if ".mlp" in nm and nm.count(".")<=4: print("    %s"%nm)
        if nm.endswith(".mlp.experts.0.down_proj"): print("    %s"%nm); break
if not ARCH_OK:
    FFN_RESULTS=dict(verdict="ARCHITEKTUR-NICHT-GEFUNDEN",prompt_id=ZIEL_ID,
                     arch_ok=False,module=[nm for nm,_ in model.named_modules()
                                           if nm.startswith("model.layers.0.mlp")])
    wc_save_all()
    print("")
    print("VERDIKT: ARCHITEKTUR-NICHT-GEFUNDEN")
    print("  Die Experten sind KEINE einzelnen Module - 'mlp.experts' hat keine")
    print("  Kinder. Das ist die gebuendelte MoE-Umsetzung: die Expertengewichte")
    print("  liegen als gestapelte Tensoren und werden mit einem Batch-Matmul")
    print("  gerechnet. Haken auf down_proj gibt es dort nicht.")
    print("  ABBRUCH HIER - die Diagnose-Zelle phase12_moe_diagnose liest den")
    print("  tatsaechlichen Aufbau aus, danach trifft der naechste Anlauf.")
    raise SystemExit(0)
# ---------------- 1  Praefixe ernten ----------------------------------------
print("")
print("="*80); print("1  PRAEFIXE ERNTEN (%d Ziehungen des Original-Prompts)"%N_ERNTE)
def zieh(text,n,startwert):
    aus=[]
    for b0 in range(0,n,CHUNK):
        b=min(CHUNK,n-b0)
        enc=tokenizer([text]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(startwert+b0)
        with torch.no_grad():
            g=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                             repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                             pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            aus.append(tokenizer.decode(g[j,enc["input_ids"].shape[1]:],
                                        skip_special_tokens=True))
    return aus
t0=time.time()
ERNTE=zieh(BASIS,N_ERNTE,SEED)
PR=ernte_praefixe(ERNTE,PLAENGE)
print("  %d Ziehungen in %.0f s | Gesamtkipprate %.1f%%"
      %(len(ERNTE),time.time()-t0,100*sum(classify_breit(t) in SWB for t in ERNTE)/len(ERNTE)))
print("  %d brauchbare Praefixe (>=4 mal, selbst nicht gekippt):"%len(PR))
for p,n in PR: print("    n=%2d  %r"%(n,p))
assert len(PR)>=2, "zu wenige Praefixe - PLAENGE senken oder N_ERNTE erhoehen"
# ---------------- 2  je Praefix die Rate messen -----------------------------
print("")
print("="*80); print("2  JE PRAEFIX ERZWINGEN UND DIE RATE MESSEN (%d Ziehungen)"%N_PRAEF)
RATE={}; ROH={}
t0=time.time()
for i,(p,_) in enumerate(PR):
    aus=zieh(BASIS+p,N_PRAEF,SEED+7919*(i+1))
    voll=[p+a for a in aus]; ROH[p]=voll
    k=sum(classify_breit(t) in SWB for t in voll); RATE[p]=k/len(voll)
    pp,lo,hi=wilson(k,len(voll))
    print("  [%d/%d] %3d/%-3d = %5.1f%% [%4.1f,%4.1f]  %r  (%.0f s)"
          %(i+1,len(PR),k,len(voll),100*pp,100*lo,100*hi,p[:34],time.time()-t0))
Y=[RATE[p] for p,_ in PR]
print("  Spannweite ueber die Praefixe: %.1f%% bis %.1f%%"%(100*min(Y),100*max(Y)))
# ---------------- 3  Zustand je Praefix aufzeichnen -------------------------
print("")
print("="*80); print("3  ROUTER UND FFN-ZWISCHENSCHICHT AUFZEICHNEN")
AUFN={}          # (schicht,experte) -> vektor, je Praefix
def zeichne_auf(text):
    """KV-Cache auf alles ausser dem letzten Token, dann GENAU EIN Token
       vorwaerts. Damit gehoert jede aufgezeichnete Zeile eindeutig zu dieser
       einen Position - kein Zurueckrechnen durch die Experten-Sammlung."""
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    fang={}
    def mach(l,e):
        def h(mod,args):
            v=args[0]
            fang[(l,e)]=v.reshape(-1,v.shape[-1])[-1].detach().float().cpu().numpy()
            return None
        return h
    hs=[EXPERTEN[l][e].register_forward_pre_hook(mach(l,e))
        for l in EXPERTEN for e in EXPERTEN[l]]
    try:
        with torch.no_grad():
            o=model(ids[:,:-1],use_cache=True)
            fang.clear()
            o2=model(ids[:,-1:],past_key_values=o.past_key_values,use_cache=True)
            letzte=o2.logits[0,-1].float().cpu().numpy()
    finally:
        for h in hs: h.remove()
    return dict(fang),letzte
ZUST={}; LOGIT0={}
t0=time.time()
for p,_ in PR:
    ZUST[p],LOGIT0[p]=zeichne_auf(BASIS+p)
print("  %d Zustaende in %.0f s"%(len(ZUST),time.time()-t0))
PAARE=sorted(set().union(*[set(z) for z in ZUST.values()]))
BREITE=len(next(iter(ZUST[PR[0][0]].values())))
print("  aktive (Schicht,Experte)-Paare ueber alle Praefixe: %d"%len(PAARE))
print("  davon in JEDEM Praefix aktiv: %d"
      %sum(1 for q in PAARE if all(q in ZUST[p] for p,_ in PR)))
A=np.zeros((len(PR),len(PAARE)*BREITE),dtype=np.float32)
for i,(p,_) in enumerate(PR):
    for j,q in enumerate(PAARE):
        if q in ZUST[p]: A[i,j*BREITE:(j+1)*BREITE]=ZUST[p][q]
nz=float((np.abs(A)>1e-3).mean())
print("  Duennbesetztheit der Zwischenschicht: %.1f%% der Einheiten |x|>1e-3"%(100*nz))
print("  (das ist die eingebaute Duennbesetztheit, um die es bei einem SAE geht)")
# ---------------- 4  Einheiten waehlen --------------------------------------
print("")
print("="*80); print("4  EINHEITEN WAEHLEN (explorativ - der Test ist Schritt 5)")
d=trenn_statistik(A,Y)
AUSW=[]; ZUFALL=[]
if d is None:
    print("  zu wenige Praefixe fuer eine Aufteilung in hohe und niedrige Haelfte")
else:
    idx=waehle_einheiten(d,K_EINH)
    AUSW=[(PAARE[i//BREITE],i%BREITE) for i in idx]
    aktiv=[i for i in range(A.shape[1]) if np.abs(A[:,i]).max()>1e-3]
    rnd=random.Random(SEED); ZUFALL=[(PAARE[i//BREITE],i%BREITE)
                                     for i in rnd.sample(aktiv,min(K_EINH,len(aktiv)))]
    print("  %d Einheiten gewaehlt (groesste positive Trennung), Werte %.2f .. %.2f"
          %(len(AUSW),d[idx[-1]],d[idx[0]]))
    print("  Kontrolle: %d zufaellige Einheiten aus den %d AKTIVEN"
          %(len(ZUFALL),len(aktiv)))
    vs=collections.Counter(q[0] for q,_ in AUSW)
    print("  Verteilung auf Schichten: %s"
          %", ".join("L%d:%d"%(l,n) for l,n in sorted(vs.items())[:12]))
# ---------------- 5  Ablation ------------------------------------------------
print("")
print("="*80); print("5  ABLATION - der eigentliche Test")
ZIEL=max(RATE,key=lambda p:RATE[p])
print("  Praefix mit der hoechsten Rate: %.1f%%  %r"%(100*RATE[ZIEL],ZIEL[:40]))
def haenge_ab(einheiten):
    nach=collections.defaultdict(list)
    for q,u in einheiten: nach[q].append(u)
    hs=[]
    for (l,e),us in nach.items():
        ui=torch.tensor(sorted(us),device=model.device)
        def mach(ui):
            def h(mod,args):
                v=args[0].clone(); v[...,ui]=0.0
                return (v,)+tuple(args[1:])
            return h
        hs.append(EXPERTEN[l][e].register_forward_pre_hook(mach(ui)))
    return hs
def mit_ablation(einheiten,n,startwert):
    hs=haenge_ab(einheiten)
    try: return zieh(BASIS+ZIEL,n,startwert)
    finally:
        for h in hs: h.remove()
K_BAS=sum(classify_breit(t) in SWB for t in ROH[ZIEL]); N_BAS=len(ROH[ZIEL])
CODE="ZU-WENIG-PRAEFIXE"; K_ABL=N_ABL=K_ZUF=N_ZUF=0
if AUSW:
    hs=haenge_ab(AUSW)
    try:
        with torch.no_grad():
            ids=tokenizer(BASIS+ZIEL,return_tensors="pt").input_ids.to(model.device)
            lg=model(ids).logits[0,-1].float().cpu().numpy()
    finally:
        for h in hs: h.remove()
    wirk=float(np.abs(lg-LOGIT0[ZIEL]).max())
    print("  WIRKUNGS-TOR: groesste Logit-Aenderung durch die Ablation %.4f"%wirk)
    assert wirk>1e-3, ("die Haken veraendern nichts - Ablation wirkungslos, "
                       "Ergebnis waere bedeutungslos")
    t0=time.time()
    a=mit_ablation(AUSW,N_PRAEF,SEED+11); K_ABL=sum(classify_breit(ZIEL+t) in SWB for t in a)
    N_ABL=len(a)
    z=mit_ablation(ZUFALL,N_PRAEF,SEED+13); K_ZUF=sum(classify_breit(ZIEL+t) in SWB for t in z)
    N_ZUF=len(z)
    print("  (%.0f s)"%(time.time()-t0))
    print("")
    print("  %-24s %10s %18s %10s"%("Bedingung","k/n","95%-Intervall","p vs Basis"))
    for nm,k,n in (("ohne Eingriff",K_BAS,N_BAS),("gewaehlte Einheiten aus",K_ABL,N_ABL),
                   ("zufaellige Einheiten aus",K_ZUF,N_ZUF)):
        pp,lo,hi=wilson(k,n)
        pv="-" if nm=="ohne Eingriff" else "%.4f"%fisher2x2(k,n-k,K_BAS,N_BAS-K_BAS)
        print("  %-24s %3d/%-4d %5.1f%% [%4.1f,%4.1f] %10s"%(nm,k,n,100*pp,100*lo,100*hi,pv))
    CODE=urteil_ffn(ARCH_OK,len(PR),K_BAS,N_BAS,K_ABL,N_ABL,K_ZUF,N_ZUF)
else:
    CODE=urteil_ffn(ARCH_OK,len(PR),K_BAS,N_BAS,0,0,0,0)
print("")
print("VERDIKT: %s"%CODE)
if CODE=="EINHEITEN-KAUSAL":
    print("  Das Nullsetzen der gewaehlten FFN-Einheiten senkt die Kipprate, das")
    print("  Nullsetzen gleich vieler zufaelliger AKTIVER Einheiten nicht. Damit")
    print("  ist die Entscheidung in der FFN-Zwischenschicht lokalisiert - in")
    print("  einer benannten, endlichen Menge von Einheiten, nicht in einer")
    print("  Richtung des ueberlagerten Residuums. Das ist der erste Griff, der")
    print("  haelt, seit die Adjektiv-Achsen gefallen sind.")
elif CODE=="UNSPEZIFISCH":
    print("  Auch zufaellige aktive Einheiten senken die Rate. Dann ist es die")
    print("  blosse Stoerung der Zwischenschicht und nicht die Auswahl - das")
    print("  Verfahren hat keine Aufloesung, egal wie klein das p ist.")
elif CODE=="KEIN-EFFEKT":
    print("  Keine der beiden Ablationen bewegt die Rate. %d Einheiten sind dann"%K_EINH)
    print("  zu wenige, oder die Entscheidung liegt nicht dort. Beides waere ein")
    print("  Befund - mit dem Wirkungs-Tor oben ist ausgeschlossen, dass die")
    print("  Haken einfach nicht gegriffen haben.")
elif CODE=="ARCHITEKTUR-NICHT-GEFUNDEN":
    print("  Die Experten-Module heissen anders als erwartet. Die Liste oben")
    print("  zeigt die tatsaechlichen Namen; damit trifft der naechste Anlauf.")
FFN_RESULTS=dict(verdict=CODE,prompt_id=ZIEL_ID,arch_ok=bool(ARCH_OK),
    n_ernte=N_ERNTE,n_praef=N_PRAEF,k_einheiten=K_EINH,praefixlaenge=PLAENGE,
    praefixe=[{"text":p,"n_ernte":n,"rate":RATE[p]} for p,n in PR],
    paare=[list(q) for q in PAARE],breite=int(BREITE),duennbesetzt=nz,
    gewaehlt=[[list(q),int(u)] for q,u in AUSW],
    zufall=[[list(q),int(u)] for q,u in ZUFALL],
    k_basis=K_BAS,n_basis=N_BAS,k_abl=K_ABL,n_abl=N_ABL,k_zufall=K_ZUF,n_zufall=N_ZUF)
wc_save("antworten_ffn",dict(prompt_id=ZIEL_ID,ernte=ERNTE,praefixe=ROH))
np.savez_compressed(os.path.join(RUN_OUT,"ffn_zustaende.npz"),
                    praefixe=np.array([p for p,_ in PR]),rate=np.array(Y),A=A,
                    paare=np.array([list(q) for q in PAARE]))
wc_save_all()
print("")
print("(Schritt 4 ist bei %d Praefixen explorativ. Schritt 5 ist der Test -"%len(PR))
print(" eine ablatierte Einheit, die die Rate bewegt, braucht keinen")
print(" Permutationstest. Zustaende und Texte liegen in Drive.)")
